In [ ]:
# Setup -- we'll run SQL right inside this notebook using sqlite3
import sqlite3
import pandas as pd

# Create an in-memory database (wipes when the kernel restarts -- perfect for teaching)
conn = sqlite3.connect(':memory:')

# Helper: run a query and return a clean DataFrame
def q(sql):
    return pd.read_sql_query(sql, conn)

# Helper: run DDL / DML statements (CREATE, INSERT, UPDATE, DELETE)
def run(sql):
    conn.executescript(sql)
    conn.commit()

print('SQL engine ready.')
print('Use q("SELECT ...") to query, run("CREATE / INSERT / ...") for everything else.')

SQL engine ready.
Use q("SELECT ...") to query, run("CREATE / INSERT / ...") for everything else.


In [ ]:
# Optional -- point the notebook at olympics.db (if it sits next to this notebook)
import os
if os.path.exists('olympics.db'):
    conn = sqlite3.connect('olympics.db')
    def q(sql): return pd.read_sql_query(sql, conn)
    def run(sql): conn.executescript(sql); conn.commit()
    print('Now querying the real Olympics database.')
    print(q('SELECT COUNT(*) AS n_athletes FROM person'))
else:
    print('olympics.db not found -- use sqliteonline for the exercises.')

Now querying the real Olympics database.
   n_athletes
0      128854


In [ ]:
# Task 1: Identify competitors who have won at least one medal in events spanning both Summer and Winter Olympics. Create a temporary table to store these competitors and their medal counts for each season, and then display the contents of this table.

conn.execute('''
DROP TABLE IF EXISTS temp_competitor_season_medals;
''')

conn.execute('''
CREATE TEMP TABLE temp_competitor_season_medals AS
SELECT
    season_medals.person_id,
    p.full_name,
    season_medals.season,
    season_medals.medal_count
FROM (
    SELECT
        gc.person_id,
        g.season,
        COUNT(*) AS medal_count
    FROM competitor_event ce
    JOIN medal m
        ON ce.medal_id = m.id
    JOIN games_competitor gc
        ON ce.competitor_id = gc.id
    JOIN games g
        ON gc.games_id = g.id
    WHERE m.medal_name != 'NA'
    GROUP BY gc.person_id, g.season
) AS season_medals
JOIN person p
    ON season_medals.person_id = p.id
WHERE season_medals.person_id IN (
    SELECT person_id
    FROM (
        SELECT
            gc.person_id,
            g.season,
            COUNT(*) AS medal_count
        FROM competitor_event ce
        JOIN medal m
            ON ce.medal_id = m.id
        JOIN games_competitor gc
            ON ce.competitor_id = gc.id
        JOIN games g
            ON gc.games_id = g.id
        WHERE m.medal_name != 'NA'
        GROUP BY gc.person_id, g.season
    ) AS both_seasons
    GROUP BY person_id
    HAVING COUNT(DISTINCT season) = 2
);
''')


In [ ]:
q('''
SELECT *
FROM temp_competitor_season_medals
ORDER BY person_id, season;
''')

,person_id,full_name,season,medal_count
0,30131,"Herbert John ""Herb"""" Drury""",Summer,1
1,30131,"Herbert John ""Herb"""" Drury""",Winter,1
2,31167,"Edward Patrick Francis ""Eddie"""" Eagan""",Summer,1
3,31167,"Edward Patrick Francis ""Eddie"""" Eagan""",Winter,1
4,42258,Gillis Emanuel Grafstrm,Summer,1
5,42258,Gillis Emanuel Grafstrm,Winter,3
6,50859,Clara Hughes,Summer,2
7,50859,Clara Hughes,Winter,4
8,53408,Walter Andreas Jakobsson,Summer,1
9,53408,Walter Andreas Jakobsson,Winter,1


In [ ]:
# Task 2: Create a temporary table to store competitors who have won medals in exactly two different sports, and then use a subquery to identify the top 3 competitors with the highest total number of medals across all sports. Display the contents of this table.

conn.execute('''
DROP TABLE IF EXISTS temp_two_sport_medalists;
''')

conn.execute('''
CREATE TEMP TABLE temp_two_sport_medalists AS
SELECT *
FROM (
    SELECT
        competitor_event.competitor_id,
        COUNT(DISTINCT event.sport_id) AS number_of_sports,
        COUNT(*) AS total_medals
    FROM competitor_event
    JOIN medal
        ON competitor_event.medal_id = medal.id
    JOIN event
        ON competitor_event.event_id = event.id
    WHERE medal.medal_name != 'NA'
    GROUP BY competitor_event.competitor_id
    HAVING COUNT(DISTINCT event.sport_id) = 2
) AS two_sport_competitors
ORDER BY total_medals DESC
LIMIT 3;
''')

In [ ]:
q('''
SELECT *
FROM temp_two_sport_medalists;
''')

,competitor_id,number_of_sports,total_medals
0,90843,2,4
1,142777,2,4
2,171999,2,4


In [ ]:
# Task 1: Retrieve the regions that have competitors who have won the highest number of medals in a single Olympic event. Use a subquery to determine the event with the highest number of medals for each competitor, and then display the top 5 regions with the highest total medals.

q('''
SELECT
    nr.region_name,
    SUM(best_event.max_medals_in_one_event) AS total_medals
FROM (
    SELECT
        competitor_event_counts.person_id,
        MAX(competitor_event_counts.medals_in_event) AS max_medals_in_one_event
    FROM (
        SELECT
            gc.person_id,
            ce.event_id,
            COUNT(*) AS medals_in_event
        FROM competitor_event ce
        JOIN medal m
            ON ce.medal_id = m.id
        JOIN games_competitor gc
            ON ce.competitor_id = gc.id
        WHERE m.medal_name != 'NA'
        GROUP BY gc.person_id, ce.event_id
    ) AS competitor_event_counts
    GROUP BY competitor_event_counts.person_id
) AS best_event
JOIN person_region pr
    ON best_event.person_id = pr.person_id
JOIN noc_region nr
    ON pr.region_id = nr.id
GROUP BY nr.region_name
ORDER BY total_medals DESC
LIMIT 5;
''')

,region_name,total_medals
0,USA,4355
1,Soviet Union,2216
2,Germany,1987
3,UK,1715
4,France,1412


In [ ]:
# Task 2: Create a temporary table to store competitors who have participated in more than three Olympic Games but have not won any medals. Retrieve and display the contents of this table, including their full names and the number of games they participated in.


conn.execute('''
DROP TABLE IF EXISTS temp_no_medals_many_games;
''')

conn.execute('''
CREATE TEMP TABLE temp_no_medals_many_games AS
SELECT
    p.full_name,
    gc.person_id,
    COUNT(DISTINCT gc.games_id) AS games_count
FROM games_competitor gc
JOIN person p
    ON gc.person_id = p.id
WHERE gc.person_id NOT IN (
    SELECT DISTINCT gc2.person_id
    FROM games_competitor gc2
    JOIN competitor_event ce
        ON gc2.id = ce.competitor_id
    JOIN medal m
        ON ce.medal_id = m.id
    WHERE m.medal_name != 'NA'
)
GROUP BY gc.person_id, p.full_name
HAVING COUNT(DISTINCT gc.games_id) > 3;
''')

In [ ]:
q('''
SELECT *
FROM temp_no_medals_many_games;
''')

,full_name,person_id,games_count
0,Hans Aasns,46,5
1,Abdihakim 'Abdi'' Abdirahman',256,4
2,Roger Abel,398,4
3,Julianne 'Anne'' Abernathy',422,5
4,Shiny Kurisingal Abraham-Wilson,512,4
...,...,...,...
1176,Wiesaw Ziemianin,134962,4
1177,Nenad Zimonji,135043,4
1178,Branko Zorko,135252,5
1179,Yelena Nikolayevna Zubrilova-Ogurtsova,135360,4
